# 03 merge(bf16) + 4bit 양자화 (배포본)

| 항목 | 값 |
|---|---|
| GPU | A100 40GB 또는 80GB (bf16 14.7B 약 29.5GB가 GPU에 올라가야 함), 고용량 RAM |
| 예상 시간 | 약 40분 (bf16 로드·merge 10분, 로컬 저장 30GB 10분, NF4 재로드·Drive 저장 9.6GB 15분) |
| 예상 컴퓨트 | 약 8~10 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 원본 59GB + merge bf16 30GB (Colab 로컬 디스크, Drive에는 NF4 9.6GB만) |
| 이어 하기 | Drive `merged_nf4/hashes.json`이 있으면 끝난 것이다. 중간에 끊기면 처음부터 다시 한다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
if not os.path.exists(f"{DIR}/bundle/notebooks/distill_common.py"):
    os.makedirs(f"{DIR}/bundle", exist_ok=True)
    !tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U "transformers>=5.9.0" peft bitsandbytes accelerate
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
OUT = dc.DRIVE / "merged_nf4"
LOCAL = "/content/merged_bf16"

In [ ]:
# 1) bf16 원본에 LoRA merge → 로컬 디스크에 저장
if not (OUT / "hashes.json").exists():
    base = AutoModelForCausalLM.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV, dtype=torch.bfloat16, device_map={"": 0})
    tok = AutoTokenizer.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV)
    merged = PeftModel.from_pretrained(base, str(dc.DRIVE / "lora" / "chosen")).merge_and_unload()
    merged.save_pretrained(LOCAL, safe_serialization=True)
    tok.save_pretrained(LOCAL)
    del base, merged; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 2) NF4(이중 양자화)로 다시 읽어 4bit로 저장 → Drive. embedding·lm_head는 bf16으로 남는다(bitsandbytes 기본)
if not (OUT / "hashes.json").exists():
    q = AutoModelForCausalLM.from_pretrained(LOCAL, quantization_config=dc.bnb_config(), dtype=torch.bfloat16, device_map={"": 0})
    q.save_pretrained(str(OUT), safe_serialization=True)
    AutoTokenizer.from_pretrained(LOCAL).save_pretrained(str(OUT))
    print("GPU 메모리(4bit 전체 GPU)", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
    del q; gc.collect(); torch.cuda.empty_cache()
    hashes = dc.dir_hashes(OUT)
    (OUT / "hashes.json").write_text(json.dumps({"source": dc.STUDENT_ID, "revision": dc.STUDENT_REV,
                                                 "adapter": dc.dir_hashes(dc.DRIVE / "lora" / "chosen"),
                                                 "files": hashes}, indent=1))
print((OUT / "hashes.json").read_text())

## repo로 가져올 파일
- `merged_nf4/hashes.json`만 → `distill/merged_nf4_hashes.json`. 가중치(약 9.6GB)는 Drive에 둔다.